<a href="https://colab.research.google.com/github/Laxmanrayka/Lucky-/blob/main/FAKE_NEWS_DECTACTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **JAI MAMA DHANI**


In [ ]:
!pip install -q transformers datasets torch pandas scikit-learn sha

ERROR: Could not find a version that satisfies the requirement sha (from versions: none)
ERROR: No matching distribution found for sha


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import torch
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments
from datasets import Dataset
import seaborn as sns
import matplotlib.pyplot as plt
#Load datasets
true=pd.read_csv('/content/True.csv',engine='python',on_bad_lines='skip')
fake=pd.read_csv('/content/Fake.csv',engine='python',on_bad_lines='skip')
#add label
true['label']=1
fake['label']=0
#combine
df=pd.concat([true,fake],ignore_index=True)
#combine title + text
df['news']=df['title']+""+df['text']
#shuffle
df=df.sample(frac=1,random_state=42).reset_index(drop=True)
#keep only necessary columns
df=df[['news','label']]
print(df['label'].value_counts())
print(df.head())


label
1    1217
0     808
Name: count, dtype: int64
                                                news  label
0   Donald Trump Whines, Swears And Talks Cocktai...      0
1   Trump Supporter In Phoenix Just Threatened Jo...      0
2  Senate's Schumer to meet with official named b...      1
3  Twitter bans ads from two Russian media outlet...      1
4  CIA's Pompeo asserts Russian meddling did not ...      1


In [ ]:
#train test split
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])
#convert to huggingface dataset
train_dataset=Dataset.from_pandas(train_df)
test_dataset=Dataset.from_pandas(test_df)
print(f"Train size{len(train_df)} Test size {len(test_df)}")

Train size1620 Test size 405


In [ ]:
#model name
model_name='bert-base-uncased'
tokenizer=AutoTokenizer.from_pretrained(model_name)
def tokenize_function(examples):
  return tokenizer(examples['news'],padding='max_length',truncation=True,max_length=512)
#tokenize
tokenized_train=train_dataset.map(tokenize_function,batched=True)
tokenized_test=test_dataset.map(tokenize_function,batched=True)
#remove unnecessary columns
tokenized_train=tokenized_train.remove_columns(['news'])
tokenized_test=tokenized_test.remove_columns(['news'])
#set format for pytorch
tokenized_train.set_format('torch')
tokeni

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1620 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

In [ ]:
# Model name
model_name = "bert-base-uncased"  # Ya "roberta-base" bhi try kar sakte ho

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['news'], padding="max_length", truncation=True, max_length=512)

# Tokenize
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns
tokenized_train = tokenized_train.remove_columns(['news'])
tokenized_test = tokenized_test.remove_columns(['news'])

# Set format for PyTorch
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              # 3 epochs enough for good accuracy
    per_device_train_batch_size=8,   # GPU memory ke hisab se adjust (T4 pe 8-16)
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

In [ ]:
# Evaluate
results = trainer.evaluate()
print(results)

# Predictions on test set
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print(classification_report(true_labels, preds, target_names=['Fake', 'Real']))

# Confusion Matrix
cm = confusion_matrix(true_labels, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Save model and tokenizer
model.save_pretrained('/content/fake_news_bert')
tokenizer.save_pretrained('/content/fake_news_bert')

In [ ]:
from transformers import pipeline

# Load fine-tuned model
classifier = pipeline("text-classification",
                      model='/content/fake_news_bert',
                      tokenizer='/content/fake_news_bert',
                      device=0)  # GPU use

# Test examples
news1 = "Scientists discover new planet with water and life signs"
news2 = "Hillary Clinton runs secret child trafficking ring from pizza shop basement"

print(classifier(news1))
print(classifier(news2))

In [ ]:
import shap
from transformers import pipeline

# SHAP explainer
explainer = shap.Explainer(classifier)

# Test a fake-looking news
shap_values = explainer(["Donald Trump announces he is actually an alien from Mars"])

# Plot
shap.plots.text(shap_values[0])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
